In [ ]:
import sys
import os
import requests
import pandas as pd
from io import BytesIO
import streamlit as st
from datetime import datetime, time
import altair as alt

st.set_page_config(layout="wide")

sys.path.append(os.path.join(os.getcwd(), "..", "data"))

#  https://1drv.ms/x/c/f6261b79731e452c/IQAWBwanD61xSKrJ_UcUYI4FAa-UyjOruqjXIwO12-5lKU4?download=1  # dw 1-24 cleaned 
#  https://1drv.ms/x/c/f6261b79731e452c/IQCf275GvfAtTq_xIZuNI9fmAU3J6I3ga_BZ1pox5YbWMD8?download=1  # dw 25-107 cleaned 


urls = [ "https://1drv.ms/x/c/f6261b79731e452c/IQBpBPUwwVtQTYrz4LgAWE6ZAfTxxxhud5AzyDm4tVY5P-0?download=1",   # Dirty 1 - 24
         "https://1drv.ms/x/c/f6261b79731e452c/IQDTm4wgClDgRKutu40S_fAUASzQrkvm2Z--Qe2whPBX1xI?download=1"
         ]   # Dirty 25 - 107

# urls = [
#     "https://1drv.ms/x/c/f6261b79731e452c/IQAWBwanD61xSKrJ_UcUYI4FAa-UyjOruqjXIwO12-5lKU4?download=1",
#     "https://1drv.ms/x/c/f6261b79731e452c/IQBpBPUwwVtQTYrz4LgAWE6ZAfTxxxhud5AzyDm4tVY5P-0?download=1" #  1-24
#         ]

sys.path.append(os.path.join(os.getcwd(), "..", "data"))

def get_data(url):
    response = requests.get(url)
    df = pd.read_excel(BytesIO(response.content), engine="openpyxl")
    # df = pd.read_csv(fd, sep='\t', header=None, skip_blank_lines=True)
    # df = df.replace(',', '', regex=True)
    # df = df.dropna(how="all")
    return df



In [246]:
df1 = get_data(urls[0])
df2 = get_data(urls[1])

In [247]:
def shape_df(df):
    # cut the first 5 rows and first column 
    df = df.iloc[5:,1:].reset_index(drop=True)
    return df

df1 = shape_df(df1)
df2 = shape_df(df2)

In [248]:
col = ['Date', 'Time', 'GPM_1', 'TOTAL_GAL_1', 'GPM_2','TOTAL_GAL_2', 'GPM_3', 'TOTAL_GAL_3', 'GPM_4', 'TOTAL_GAL_4', 'GPM_5','TOTAL_GAL_5', 'GPM_6', 'TOTAL_GAL_6', 'GPM_7', 'TOTAL_GAL_7', 'GPM_8','TOTAL_GAL_8', 'GPM_9', 'TOTAL_GAL_9', 'GPM_10', 'TOTAL_GAL_10', 'GPM_11','TOTAL_GAL_11', 'GPM_12', 'TOTAL_GAL_12', 'GPM_13', 'TOTAL_GAL_13','GPM_14', 'TOTAL_GAL_14', 'GPM_15', 'TOTAL_GAL_15', 'GPM_16','TOTAL_GAL_16', 'GPM_17', 'TOTAL_GAL_17', 'GPM_18', 'TOTAL_GAL_18','GPM_19', 'TOTAL_GAL_19', 'GPM_20', 'TOTAL_GAL_20', 'GPM_21','TOTAL_GAL_21', 'GPM_22', 'TOTAL_GAL_22', 'GPM_23', 'TOTAL_GAL_23','GPM_24', 'TOTAL_GAL_24', 'comments', 'datetime']
col2 = ['Date', 'Time','GPM_25', 'TOTAL_GAL_25', 'GPM_26','TOTAL_GAL_26','GPM_27', 'TOTAL_GAL_27', 'GPM_28','TOTAL_GAL_28', 'GPM_29', 'TOTAL_GAL_29', 'GPM_30','TOTAL_GAL_30', 'GPM_31', 'TOTAL_GAL_31', 'GPM_32', 'TOTAL_GAL_32', 'GPM_33','TOTAL_GAL_33', 'GPM_34', 'TOTAL_GAL_34', 'GPM_35', 'TOTAL_GAL_35', 'GPM_36','TOTAL_GAL_36', 'GPM_37', 'TOTAL_GAL_37', 'GPM_38', 'TOTAL_GAL_38', 'GPM_39','TOTAL_GAL_39', 'GPM_40', 'TOTAL_GAL_40', 'GPM_41', 'TOTAL_GAL_41','GPM_42', 'TOTAL_GAL_42', 'GPM_43', 'TOTAL_GAL_43', 'GPM_44','TOTAL_GAL_44', 'GPM_45', 'TOTAL_GAL_45', 'GPM_101', 'TOTAL_GAL_101','GPM_102', 'TOTAL_GAL_102', 'GPM_103', 'TOTAL_GAL_103', 'GPM_104','TOTAL_GAL_104', 'GPM_105', 'TOTAL_GAL_105', 'GPM_106', 'TOTAL_GAL_106','GPM_107', 'TOTAL_GAL_107', 'comments', 'datetime']

def trim_frame(df, new_col):
    df = df.iloc[:, :len(new_col)] 
    df.columns = new_col
    return df

df1 = trim_frame(df1, col)
df2 = trim_frame(df2, col2)


In [249]:
def clean_col(df):
    # strip all columns with "Unnamed" string
    for col in df.columns:
        if "Unnamed" in str(col):
            df = df.drop(columns=[str(col)], errors='ignore')
    return df

df1 = clean_col(df1)
df2 = clean_col(df2)

In [250]:
def fix_time(df):
    mask = df['Time'].apply(lambda x: isinstance(x, datetime))
    df.loc[mask, 'Time'] = df.loc[mask, 'Time'].apply(lambda x: x.time())
    return df

df1 = fix_time(df1)
df2 = fix_time(df2)


In [251]:
def make_datetime_col(df):
    df['datetime'] = df.apply(
        lambda row: datetime.combine(row['Date'].date(), row['Time']),
        axis=1
    )
    return df
df1 = make_datetime_col(df1)
df2 = make_datetime_col(df2)


In [252]:
def fix_space(df):
    df.columns = df.columns.str.replace('.', '_', regex=False)
    df.columns = df.columns.str.replace(' ', '_', regex=False)  # replace spaces
    return df

df1 = fix_space(df1)
df2 = fix_space(df2)


In [253]:
def drop_DT(df):
    # strip all columns with "Unnamed" string
    df = df.drop(columns=['Date', 'Time'], errors='ignore')
    return df
df1 = drop_DT(df1)
df2 = drop_DT(df2)

In [254]:
df_merged = pd.merge(df1, df2, on='datetime', how='outer', suffixes=('_1', '_2'))
# df_merged.drop(columns=['comments_1', 'comments_2'], inplace=True)

In [255]:
df_merged.sort_values('datetime', inplace=True)
df_merged.reset_index(drop=True, inplace=True)

In [257]:
df_merged.to_excel('sorted3.xlsx')

In [187]:
df_combined = pd.concat([df1, df2], ignore_index=True)

In [33]:
df_long = pd.wide_to_long(
    df2,
    stubnames=["GPM", "TOTAL GAL"],
    i="row_id",  
    j="meter",
    sep="_",
    suffix="\\d+"
).reset_index()

In [35]:
print(df2.tail())

      row_id                 Date      Time GPM_0 TOTAL GAL_0 GPM_1  \
2286    2286  2026-04-01 00:00:00  15:00:00  28.9     5601795  23.6   
2287    2287  2026-04-01 00:00:00  17:00:00  28.8     5605349    24   
2288    2288  2026-04-01 00:00:00  19:00:00  29.1     5608777  24.1   
2289    2289  2026-04-01 00:00:00  21:00:00  29.1     5612265    25   
2290    2290  2026-04-01 00:00:00  23:00:00  29.1     5616056  23.2   

     TOTAL GAL_1 GPM_2 TOTAL GAL_2 GPM_3  ... GPM_20 TOTAL GAL_20 GPM_21  \
2286     4753736  18.5     3594162   9.8  ...    3.1       214295   0.05   
2287     4756630  16.4     3596278   9.7  ...   0.22       214320   0.06   
2288     4759400  18.3     3598325   9.8  ...   0.21       214352   0.06   
2289     4762272  17.6     3600424   9.7  ...    0.2       214387   0.12   
2290     4765384  18.6     3602731    10  ...   0.21       214418   0.05   

     TOTAL GAL_21 GPM_22 TOTAL GAL_22 GPM_23 TOTAL GAL_23   comments  \
2286        68948    2.3       836036      3

In [322]:
print(df2[df2['Time'].isna()])

Empty DataFrame
Columns: [Date, Time, GPM, TOTAL GAL, GPM.1, TOTAL GAL.1, GPM.2, TOTAL GAL.2, GPM.3, TOTAL GAL.3, GPM.4, TOTAL GAL.4, GPM.5, TOTAL GAL.5, GPM.6, TOTAL GAL.6, GPM.7, TOTAL GAL.7, GPM.8, TOTAL GAL.8, GPM.9, TOTAL GAL.9, GPM.10, TOTAL GAL.10, GPM.11, TOTAL GAL.11, GPM.12, TOTAL GAL.12, GPM.13, TOTAL GAL.13, GPM.14, TOTAL GAL.14, GPM.15, TOTAL GAL.15, GPM.16, TOTAL GAL.16, GPM.17, TOTAL GAL.17, GPM.18, TOTAL GAL.18, GPM.19, TOTAL GAL.19, GPM.20, TOTAL GAL.20, GPM.21, TOTAL GAL.21, GPM.22, TOTAL GAL.22, GPM.23, TOTAL GAL.23, comments, Unnamed: 51, Unnamed: 52, Unnamed: 53, Unnamed: 54, Unnamed: 55, Unnamed: 56, Unnamed: 57, Unnamed: 58, Unnamed: 59, Unnamed: 60, Unnamed: 61, Unnamed: 62, Unnamed: 63, Unnamed: 64, Unnamed: 65, Unnamed: 66, Unnamed: 67, Unnamed: 68, Unnamed: 69, Unnamed: 70, Unnamed: 71, Unnamed: 72, Unnamed: 73, Unnamed: 74, Unnamed: 75, Unnamed: 76]
Index: []

[0 rows x 77 columns]


In [285]:
print((df2['Time'].iloc[501]))


1900-01-12 00:00:00


In [291]:
import pandas as pd
from datetime import datetime, time

def extract_time(val):
    if pd.isna(val):
        return None
    if isinstance(val, datetime):
        return val.time()
    if isinstance(val, time):
        return val
    return None

# df2['Time'].iloc[501] = df2['Time'].iloc[501].apply(extract_time)
x = str(df2['Time'].iloc[501]).split(" ")
y = pd.to_datetime(x[1]).time()
print(y)
print(type(y))

00:00:00
<class 'datetime.time'>


In [305]:
from datetime import datetime

mask = df2['Time'].apply(lambda x: isinstance(x, datetime))

df2.loc[mask, 'Time'] = df2.loc[mask, 'Time'].apply(lambda x: x.time())

In [310]:
for val in df2["Time"]:
    if isinstance(val, datetime):
        print(val)